# 🧪 W4-D5 概念实验：chunk 大小、overlap、top-K 三个旋钮怎么拧？

> 配套阅读：`ima/第4周-Day5-RAG实战架构设计与优化.md`（生产级管道、分块策略对比表、优化速查表在那边）
>
> Demo 级 RAG 上不了线，问题往往出在三个"看起来人畜无害"的参数上。这个 notebook 用模拟实验回答：
> 1. **chunk_size**：切太大/太小，recall 分别怎么死？（附 overlap 的保险作用）
> 2. **top-K**：K 越大召回越高，为什么答案质量反而先升后降？（"lost in the middle" 模拟）
> 3. **overlap 的成本账**：多花 40% 索引体积，买回多少 recall？
>
> 实验环境：纯 numpy 模拟检索，无真实模型。

## 实验 1：chunk 宽度扫描 —— 太切碎事实，太大稀释信号

构造一份"每句话一个产品事实"的模拟文档，用固定宽度滑窗切割，
检索器 = 字符 bigram 余弦（W4-D2 同款）。**命中标准很严格：Top-1 chunk 必须同时包含产品名和价格。**

- 窗口太小：事实被拦腰斩断（W4-D1 实验3 的放大版）；
- 窗口太大：一个 chunk 混进多个产品，bigram 信号被稀释，还会把错误产品塞进 prompt。

In [ ]:
import re
import numpy as np

rng = np.random.default_rng(42)

PRODUCTS = [
    ("杨枝甘露", "泰国椰浆", 22), ("芒果双皮奶", "顺德牛奶", 18), ("芋泥波波冰", "荔浦芋头", 17),
    ("西瓜冰", "麒麟西瓜", 16), ("双皮奶", "水牛奶", 15), ("红豆沙", "云南红豆", 12),
    ("桂花酸梅汤", "乌梅山楂", 10), ("椰汁西米露", "海南椰汁", 14),
]
FILLER = ["门店营业时间为早十点到晚十点。", "扫码点单可享会员积分。", "本店支持预约外送服务。", "节假日正常营业欢迎光临。"]

facts = [f"{n}选用{m}制作，售价{p}元。" for n, m, p in PRODUCTS]
doc = "".join(rng.permutation(FILLER + FILLER)[:3]) + "".join(
    [facts[i] + FILLER[i % 4] for i in range(8)]
) + FILLER[0]

def bigrams(t):
    t = re.sub(r"[，。、；：\s]", "", t)
    return [t[i:i+2] for i in range(len(t) - 1)]

def hit_rate(width, overlap):
    step = max(1, int(width * (1 - overlap)))
    chunks = [doc[i:i+width] for i in range(0, len(doc), step) if i + width * 0.5 <= len(doc) or i == 0]
    vocab = sorted({g for c in chunks for g in bigrams(c)})
    gidx = {g: i for i, g in enumerate(vocab)}
    def vec(t):
        v = np.zeros(len(vocab))
        for g in bigrams(t):
            if g in gidx:
                v[gidx[g]] += 1
        n = np.linalg.norm(v)
        return v / n if n > 0 else v
    M = np.stack([vec(c) for c in chunks])
    hits = 0
    for name, _, price in PRODUCTS:
        q = vec(f"{name}多少钱")
        best = chunks[int(np.argmax(M @ q))]
        if name in best and f"{price}元" in best:
            hits += 1
    return hits / len(PRODUCTS), len(chunks)

widths = [12, 18, 24, 32, 45, 65, 95, 130]
SWEEP = {}
for ov in (0.0, 0.3):
    SWEEP[ov] = [hit_rate(w, ov) for w in widths]
    print(f"overlap={ov:.0%}：", "  ".join(f"w{w}:{r*100:.0f}%" for w, (r, _) in zip(widths, SWEEP[ov])))

print("\n读数：窗口太小→事实被切断全灭；适中(≈一条完整事实)最优；")
print("     过大→单chunk多产品，查询信号被稀释，且错误产品会混进上下文。")

## 实验 2：top-K 的甜蜜点 —— 召回越多，答案不一定越好

模拟 40 个 chunk（6 个相关、34 个噪声），检索分数有重叠分布。
K 增大：recall 单调上升、precision 下降是常识；关键是**"lost in the middle"**——
LLM 对长上下文中间位置的注意力最弱（U 形），塞得越多，相关 chunk 反而被噪声淹没。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(7)
N_REL = 6
scores_rel = rng.normal(0.72, 0.08, N_REL)     # 相关 chunk 的检索分
scores_noise = rng.normal(0.50, 0.13, 40 - N_REL)
all_scores = np.concatenate([scores_rel, scores_noise])
is_rel = np.concatenate([np.ones(N_REL), np.zeros(40 - N_REL)])
order = np.argsort(-all_scores)
rank_rel = np.where(is_rel[order] == 1)[0] + 1   # 相关 chunk 的名次

ks = np.arange(1, 21)
recall = [np.mean(rank_rel <= k) for k in ks]
precision = [N_REL / k if k <= N_REL else np.sum(rank_rel <= k) / k for k in ks]

def usable(k):
    # lost-in-the-middle：位置权重 U 形——首位/末位 0.95，正中间只有 0.45
    if k == 1:
        return 0.95
    w = 0.45 + 0.5 * np.abs(np.cos(np.linspace(0, np.pi, k)))
    p_miss = 1.0
    for r in rank_rel[rank_rel <= k]:
        p_miss *= (1 - w[r - 1])          # 所有相关 chunk 都没被'利用'的概率
    return 1 - p_miss

eff = [usable(k) for k in ks]
best_k = ks[int(np.argmax(eff))]

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(ks, recall, "o-", label="Recall@K（检索到的）", color="#4f81bd")
ax.plot(ks, precision, "s-", label="Precision@K（ retrieved 里相关的占比）", color="#9bbb59")
ax.plot(ks, eff, "^-", label="有效利用率（含 lost-in-the-middle 衰减）", color="#c0504d")
ax.axvline(best_k, ls="--", color="gray")
ax.annotate(f"甜蜜点 K={best_k}", (best_k, 0.08), ha="center", fontsize=10, color="gray")
ax.set_xlabel("Top-K（塞进 prompt 的 chunk 数）")
ax.set_ylabel("指标")
ax.set_title("K 越大召回越高，但 LLM 的有效利用率先升后降")
ax.legend()
plt.tight_layout()
plt.show()

print(f"实测甜蜜点 K={best_k}：再往上堆 chunk，召回收益趋于 0，噪声和中间遗忘持续变差。")
print("对应 md 里的结论：'检索结果越多越好'是误区，K 要对着评测集调。")

## 实验 3：overlap 的成本账 —— 用 40% 索引换多少 recall？

拿实验 1 的扫描结果算账：overlap 30% 意味着 step 缩短 → chunk 数 ≈ +43%。
看它在对每个宽度档位各买回了多少 recall，再决定要不要开。

In [ ]:
import numpy as np   # 复用实验1的 widths / SWEEP

print(f"{'宽度':>6}{'无overlap':>12}{'有overlap30%':>14}{'recall提升':>12}{'chunk数 0%→30%':>18}")
gains = []
for w, (r0, n0), (r3, n3) in zip(widths, SWEEP[0.0], SWEEP[0.3]):
    gain = r3 - r0
    gains.append(gain)
    print(f"{w:>8}{r0:>10.0%}{r3:>13.0%}{gain:>+11.0%}{n0:>10}→{n3:<6}")

print(f"\n平均 recall 提升：{np.mean(gains):+.0%}；平均索引膨胀：{np.mean([b/a for (_,a),(_,b) in zip(SWEEP[0.0],SWEEP[0.3])])-1:+.0%}")
print("\n结论（对照 md 的分块策略表）：")
print("  1) chunk_size 先对齐'一个完整事实'的长度（本实验约 24-32 字），别拍脑袋；")
print("  2) overlap 是便宜的保险：小 chunk 受益最大，大 chunk 本身就跨多个事实，收益趋零；")
print("  3) top-K 对着'有效利用率'调，不是对着 recall 调（实验2）。")

## 结论

| 旋钮 | 拧错了会怎样 | 实验证据 |
|---|---|---|
| chunk_size | 太小→事实被切断；太大→信号稀释+混入无关产品 | 实验1：宽度扫描呈倒U形 |
| overlap | 不开→边界事实丢失 | 实验1/3：+43%索引换回边界事实 |
| top-K | 盲目调大→噪声+中间遗忘 | 实验2：有效利用率在 K≈5 见顶 |

→ 深入阅读：`ima/第4周-Day5-RAG实战架构设计与优化.md`（生产管道五层架构、上下文优化、生成约束、性能速查表）